# NB09 — Musanga cover versus deforestation age on native disturbance grids
V5 · CPU · Run 00–03 sequentially. Replaces the previous 50 m exploratory notebook (retained in Git history at bac1092ab241bb4c2f4e9e89083995ac0e520c57). No HLS extraction, model training or NB08 dependency. Uses NB07 final binary canopy maps, the original ROI eroded 100 m, Hansen v1.13 and JRC TMF v2025. Each dataset retains its own actual native CRS, pixel dimensions and lattice origin, checked from Earth Engine source bands (not mosaic default projection). UAV cover estimated by dense 30×30 subpixel sampling per native pixel (~1 m); ≥90% valid support inside ROI. Overlapping mosaics: select highest support, deterministic name tie-break; no double counting. Dates strictly before UAV survey calendar year. No detected event is not age zero or proof of undisturbed forest. JRC deforestation age = elapsed time since latest transition from mapped forest (1,2,4) to deforested land (3); persistent class 3 does not reset the clock. Regrowth age = elapsed time since onset of current continuous class 4 episode following mapped deforestation (3). Other transitions and left-censored histories are reported, not assigned invented ages. Retrospective classifications can use later imagery. Spatial chronosequence, not longitudinal succession or causal evidence.
Sources: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2025_v1_13 ; https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee .

**Extension:** after completing 00–03, run 04 for native-grid annual age summaries and latest mapped degradation onset; 05 for a separate 50 m HLS spectral-level/dynamics ablation using existing matched validation data. Fresh runtime after a successful native run: 00 → 01 → 04 → 05 (skip 02–03). No UAV reaggregation or HLS download. 04 requires a small repeat JRC-only history extraction because the native tables did not store degradation dates. Original native tables and outputs stay unchanged.



**Final HLS analysis (06):** use the existing NB08bis annual HLS rasters on their already aligned 30 m UTM lattice, directly aggregate UAV cover to those pixel footprints and analyze spectral levels and dynamics. No Earth Engine calls or new HLS downloads in 06. This is a common 30 m export lattice, not a claim of exact native HLS scene grids. Fresh runtime: 00 (now offline by default) → 06A → 06B; then optionally 05 for the existing matched 50 m model comparison. Hansen/JRC native-grid results remain unchanged. Earlier native HLS experiments are retained in Git history.

**Scientific reporting (07):** run 00 → 07 for one concise Word document, overwritten each time. Select the most recent compatible completed sources. No extraction or new model training.


In [ ]:
#@title 00 — Mount Drive, verify UAV inputs and study region
%pip -q install earthengine-api rasterio geopandas pyarrow scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,math,time
from collections import defaultdict
from contextlib import ExitStack
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio,ee
from rasterio import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert ROOT is not None
SITES=['Baego','MOS','Yangambi','Yoko'];YEARS=dict(Baego=2023,MOS=2024,Yangambi=2024,Yoko=2020)
PROJECT='musanga-511110';CRS='EPSG:32635';SUPPORT=.90;SUB=30;TILE=64
GID='UMD/hansen/global_forest_change_2025_v1_13';JID='projects/JRC/TMF/v1_2025/AnnualChanges'
FORCE_REBUILD=False
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,x):
 tmp=p.with_suffix('.partial');tmp.write_text(json.dumps(x,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(p,x):
 tmp=p.with_suffix('.partial');x.to_parquet(tmp,index=False);os.replace(tmp,p)
NB07=ROOT/'Outputs/07_RGB_unified_production_v5';M=json.loads((NB07/'07_production_mosaic_manifest.json').read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
MOS=defaultdict(list);meta=[]
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 assert p.is_file(),f'Missing NB07 mosaic: {p}'
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest'] and ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5')
  b=list(ds.bounds)
 MOS[item['site']].append((item['name'],p,b));meta.append(dict(name=item['name'],site=item['site'],signature=item['digest'],size=p.stat().st_size,bounds=b))
assert set(MOS)==set(SITES)
for s in SITES:MOS[s].sort(key=lambda x:x[0])
raw=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
rp=Path(raw['roi'])
if not rp.is_file():rp=ROOT.parent/'Data/Ortho_RGB'/rp.name
assert rp.is_file()
roi=unary_union(gpd.read_file(rp).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
GEOM={s:roi.intersection(unary_union([box(*b) for _,_,b in MOS[s]])) for s in SITES}
assert all(not g.is_empty for g in GEOM.values())
def init_ee():
 try:ee.Initialize(project=PROJECT)
 except Exception:ee.Authenticate();ee.Initialize(project=PROJECT)
print('Verified six NB07 mosaics; no new UAV inference.')


In [ ]:
#@title 01 — Obtain and verify the original Hansen and JRC lattices
init_ee()
G=ee.Image(GID);J=ee.ImageCollection(JID)
def projinfo(img,band):
 p=img.select(band).projection().getInfo()
 a=Affine(*p['transform']);assert abs(a.b)<1e-12 and abs(a.d)<1e-12 and a.a>0 and a.e<0,p
 return p['crs'],a
def samegrid(p,q):
 c,a=p;d,b=q
 if c!=d or not np.allclose([a.a,a.b,a.d,a.e],[b.a,b.b,b.d,b.e],rtol=1e-9,atol=1e-12):return False
 x,y=(~a)*(b.c,b.f)
 return abs(x-round(x))<1e-5 and abs(y-round(y))<1e-5
PG=projinfo(G,'lossyear')
for band in ['datamask','treecover2000']:assert samegrid(PG,projinfo(G,band))
GRIDS={};IMAGES={};REGIONS={};TILES={}
for s in SITES:
 y=YEARS[s];g=GEOM[s]
 lonlat=gpd.GeoSeries([g],crs=CRS).to_crs('EPSG:4326').iloc[0]
 region=ee.Geometry.Rectangle(list(lonlat.bounds),proj='EPSG:4326',geodesic=False)
 jc=J.filterBounds(region).sort('system:index');ids=jc.aggregate_array('system:index').getInfo()
 assert ids,f'No JRC source tiles at {s}'
 bands=[f'Dec{k}' for k in range(1990,y)]
 js=jc.toList(len(ids));pj=projinfo(ee.Image(js.get(0)),bands[0])
 # Verify every selected annual source band on every intersecting original tile.
 for i in range(len(ids)):
  src=ee.Image(js.get(i));info=src.select(bands).getInfo()['bands']
  for b in info:
   assert samegrid(pj,(b['crs'],Affine(*b['crs_transform']))),(s,b)
 # Use source projection rather than default one-degree mosaic projection.
 jimg=jc.select(bands).mosaic().setDefaultProjection(pj[0],list(pj[1])[:6]).unmask(255).toUint8()
 land=G.select('datamask');loss=G.select('lossyear').unmask(0)
 gimg=ee.Image.cat([land.unmask(255),loss.updateMask(land.eq(1)).unmask(255),G.select('treecover2000').unmask(255)]).toUint8()
 for tag,p,img in [('Hansen',PG,gimg),('JRC',pj,jimg)]:
  c,a=p;GRIDS[s,tag]=p;IMAGES[s,tag]=img
  geom=gpd.GeoSeries([g],crs=CRS).to_crs(c).iloc[0];REGIONS[s,tag]=geom
  x0,y0,x1,y1=geom.bounds
  cl,rt=(~a)*(x0,y1);cr,rb=(~a)*(x1,y0)
  tiles=[]
  for row in range(math.floor(rt/TILE)*TILE,math.ceil(rb/TILE)*TILE,TILE):
   for col in range(math.floor(cl/TILE)*TILE,math.ceil(cr/TILE)*TILE,TILE):
    l,t=a*(col,row);r,b=a*(col+TILE,row+TILE)
    if geom.intersects(box(l,b,r,t)):tiles.append((row,col))
  assert tiles;TILES[s,tag]=tiles
  print(s,tag,c,list(a)[:6],len(tiles),'tiles')
CFG=dict(version='native_ecology_v1',sources=[GID,JID],years=YEARS,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),mosaics=meta,model=M['model_sha'],policy=M['policy_sha'],support=SUPPORT,sub=SUB,tile=TILE,grids={f'{s}_{t}':[c,list(a)[:6]] for (s,t),(c,a) in GRIDS.items()})
OUT=ROOT/'Outputs/09_ecology_v5'/digest(CFG)[:20];OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/'tiles';CACHE.mkdir(exist_ok=True);atomic_json(OUT/'config.json',CFG)
print('Native-grid output:',OUT)


In [ ]:
#@title 02 — Aggregate UAV cover directly to native pixels and attach disturbance histories
def native_tile(s,tag,row,col,stack):
 c,a=GRIDS[s,tag];tr=a*Affine.translation(col,row)
 # Approximate area integration by equally spaced subpixel centres within each native footprint.
 hi=tr*Affine.scale(1/SUB,1/SUB);size=TILE*SUB
 inside=rasterize([(REGIONS[s,tag],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(TILE,SUB,TILE,SUB).sum((1,3))
 best=np.zeros((TILE,TILE),dtype=np.int32);positive=best.copy();source=np.full(best.shape,-1,dtype=np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  ds=stack[name]
  with WarpedVRT(ds,crs=c,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:
   z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1))
  better=nv>best;best[better]=nv[better];positive[better]=npix[better];source[better]=i
 keep=best>=math.ceil(SUPPORT*SUB*SUB)
 if not keep.any():return pd.DataFrame()
 # Explicit PixelGrid affine; Earth Engine returns structured NPY, not a resampled arbitrary 30m UTM tile.
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:
   arr=ee.data.computePixels(dict(expression=IMAGES[s,tag],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 assert arr.shape==(TILE,TILE)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 d=pd.DataFrame(dict(site=s,product=tag,row=rr+row,col=cc+col,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(SUB*SUB),cover_pp=100*positive[keep]/best[keep],mosaic=[MOS[s][i][0] for i in source[keep]]))
 y=YEARS[s]
 if tag=='Hansen':
  assert set(arr.dtype.names)=={'datamask','lossyear','treecover2000'},arr.dtype.names
  mask=arr['datamask'][keep];loss=arr['lossyear'][keep].astype(int);tc=arr['treecover2000'][keep]
  ok=(mask==1)&(loss>=0)&(loss<=25)
  prior=ok&(loss>0)&(2000+loss<y)
  d['history_valid']=ok;d['loss_year']=np.where(prior,2000+loss,np.nan)
  d['age_since_loss']=y-d.loss_year
  d['loss_status']=np.select([~ok,prior,ok&(loss>0)&(2000+loss>=y)],['invalid','prior loss','survey year or later loss'],default='no detected loss')
  d['treecover2000']=np.where(tc<=100,tc,np.nan)
 else:
  years=np.arange(1990,y);names=[f'Dec{k}' for k in years]
  assert set(arr.dtype.names)==set(names),arr.dtype.names
  hist=np.stack([arr[b][keep] for b in names],axis=0)
  ok=np.isin(hist,np.arange(1,7)).all(axis=0)
  # Only genuine transitions from forest to class 3, not each year spent deforested.
  entries=np.isin(hist[:-1],[1,2,4])&(hist[1:]==3)
  latest=np.max(np.where(entries,years[1:,None],0),axis=0)
  # Current uninterrupted class 4 episode, with directly preceding deforested state.
  run=np.cumprod((hist[::-1]==4).astype(np.int16),axis=0).sum(axis=0)
  onset=len(years)-run;prev=hist[np.maximum(onset-1,0),np.arange(len(d))]
  confirmed=ok&(run>0)&(onset>0)&(prev==3)
  d['history_valid']=ok;d['jrc_state']=np.where(ok,hist[-1],np.nan)
  d['deforestation_year']=np.where(ok&(latest>0),latest,np.nan)
  d['age_since_deforestation']=y-d.deforestation_year
  d['regrowth_onset_year']=np.where(confirmed,years[0]+onset,np.nan)
  d['age_since_regrowth']=y-d.regrowth_onset_year
  d['deforestation_status']=np.select([~ok,latest>0,(hist==3).any(axis=0)],['invalid','dated forest-to-deforested transition','deforested state without dated forest transition'],default='no detected deforestation')
  d['regrowth_status']=np.select([~ok,confirmed,run>0],['invalid','dated post-deforestation regrowth','other or left-censored regrowth'],default='not currently regrowth')
 return d
all_tables={}
for s in SITES:
 with ExitStack() as es:
  stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
  for tag in ['Hansen','JRC']:
   chunks=[]
   for i,(row,col) in enumerate(TILES[s,tag]):
    p=CACHE/f'{s}_{tag}_{row}_{col}.parquet'
    if p.is_file() and not FORCE_REBUILD:d=pd.read_parquet(p)
    else:
     d=native_tile(s,tag,row,col,stack);atomic_parquet(p,d)
    if len(d):chunks.append(d)
    print(s,tag,i+1,'/',len(TILES[s,tag]),'pixels',len(d),flush=True)
   assert chunks,(s,tag)
   all_tables[s,tag]=pd.concat(chunks,ignore_index=True)
for tag in ['Hansen','JRC']:
 d=pd.concat([all_tables[s,tag] for s in SITES],ignore_index=True)
 assert not d.duplicated(['site','row','col']).any()
 assert d.cover_pp.between(0,100).all() and d.support.ge(SUPPORT).all()
 atomic_parquet(OUT/f'{tag}_native_pixels.parquet',d)
 display(d.groupby('site').agg(n=('cover_pp','size'),valid_history=('history_valid','sum'),mean_support=('support','mean')))


In [ ]:
#@title 03 — Four simple figures: raw pixel-level cover against age and JRC state
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
rows=[]
for d,col,label in [(H,'age_since_loss','Years since Hansen detected tree-cover loss'),(J9,'age_since_deforestation','Years since JRC forest-to-deforested transition'),(J9,'age_since_regrowth','Years since onset of JRC post-deforestation regrowth')]:
 fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
 maxage=max(1,d[col].max()) if d[col].notna().any() else 1
 for s,ax in zip(SITES,axs.ravel()):
  g=d[d.site.eq(s)];q=g[g.history_valid&g[col].notna()]
  x=q[col];y=q.cover_pp
  rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
  ax.scatter(x,y,s=5,alpha=.18,rasterized=True)
  ax.set(title=f'{s} | n={len(q):,}',xlabel=label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  rows.append(dict(site=s,variable=col,n=len(q),n_total=len(g),n_valid_history=int(g.history_valid.sum()),rho=rho))
 fig.savefig(OUT/f'{col}.png',dpi=220);plt.show();plt.close(fig)
names={1:'Undisturbed TMF',2:'Degraded TMF',3:'Deforested land',4:'Regrowth',5:'Water',6:'Other land cover'}
fig,axs=plt.subplots(2,2,figsize=(12,8),sharey=True,layout='constrained')
for s,ax in zip(SITES,axs.ravel()):
 g=J9[J9.site.eq(s)&J9.history_valid];groups=[];labels=[]
 for k,v in names.items():
  q=g[g.jrc_state.eq(k)].cover_pp
  if len(q):groups.append(q.to_numpy());labels.append(f'{v}\n(n={len(q):,})')
 if groups:ax.boxplot(groups,tick_labels=labels,showfliers=True)
 ax.tick_params(axis='x',labelrotation=30,labelsize=8);ax.set(title=s,ylabel='UAV Musanga cover (%)',ylim=(0,100))
fig.savefig(OUT/'JRC_state_cover.png',dpi=220);plt.show();plt.close(fig)
summary=pd.DataFrame(rows);summary.to_csv(OUT/'age_scatter_summary.csv',index=False);display(summary)
for d,col in [(H,'loss_status'),(J9,'deforestation_status'),(J9,'regrowth_status'),(J9,'jrc_state')]:
 q=d.groupby(['site',col],dropna=False).cover_pp.agg(n='size',mean_cover='mean',median_cover='median').reset_index()
 q.to_csv(OUT/f'{col}_summary.csv',index=False);display(q.round(2))
atomic_json(OUT/'completion.json',dict(complete=True,config=CFG,n_Hansen=len(H),n_JRC=len(J9),outputs=sorted(p.name for p in OUT.iterdir())))
print('NB09 NATIVE GRID COMPLETE',OUT)


## 04 — Annual age distributions and time since degradation
One point per integer age and forest: mean, median, interquartile range, n and number of 500 m spatial blocks. All dated pixels retained; sparse ages explicitly marked. No smoothing or fitted curve. IQR describes variability, not a confidence interval. JRC degradation age is elapsed time since latest transition from mapped undisturbed or regrowing forest (1 or 4) into class 2; persistent class 2 does not reset age. Re-entry after a nonforest state is not treated as confirmed forest degradation. Pixels starting in class 2 are left-censored unless a later dated entry occurs. Re-read JRC source histories on the exact native grid; re-use stored UAV fractions and verify agreement with previous latest-state and deforestation-year fields. No UAV inference/aggregation.

In [ ]:
#@title 04 — Native-grid annual cover statistics and latest JRC degradation age
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
assert (OUT/'completion.json').is_file() and json.loads((OUT/'completion.json').read_text())['complete']
O4=OUT/'04_annual_ages_and_degradation_v1';O4.mkdir(exist_ok=True)
DGC=O4/'jrc_tiles';DGC.mkdir(exist_ok=True)
def degradation_tile(s,row,col,g):
 c,a=GRIDS[s,'JRC'];tr=a*Affine.translation(col,row)
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:arr=ee.data.computePixels(dict(expression=IMAGES[s,'JRC'],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 years=np.arange(1990,YEARS[s]);bands=[f'Dec{k}' for k in years]
 assert arr.shape==(TILE,TILE) and set(arr.dtype.names)==set(bands)
 rr=g.row.to_numpy(int)-row;cc=g.col.to_numpy(int)-col
 hist=np.stack([arr[b][rr,cc] for b in bands])
 ok=np.isin(hist,np.arange(1,7)).all(axis=0)
 assert np.array_equal(ok,g.history_valid.to_numpy())
 assert np.array_equal(np.where(ok,hist[-1],np.nan),g.jrc_state.to_numpy(),equal_nan=True)
 defo=np.max(np.where(np.isin(hist[:-1],[1,2,4])&(hist[1:]==3),years[1:,None],0),axis=0)
 assert np.array_equal(np.where(ok&(defo>0),defo,np.nan),g.deforestation_year.to_numpy(),equal_nan=True)
 entry=np.isin(hist[:-1],[1,4])&(hist[1:]==2)
 latest=np.max(np.where(entry,years[1:,None],0),axis=0)
 d=g[['site','row','col']].copy()
 d['degradation_year']=np.where(ok&(latest>0),latest,np.nan)
 d['age_since_degradation']=YEARS[s]-d.degradation_year
 d['n_degradation_entries']=np.where(ok,entry.sum(axis=0),np.nan)
 d['degradation_status']=np.select([~ok,latest>0,(hist==2).any(axis=0)],['invalid','dated forest-to-degraded transition','degraded state without dated forest transition'],default='no detected degradation')
 return d
parts=[]
for (s,row,col),g in J9.assign(tile_row=(J9.row//TILE)*TILE,tile_col=(J9.col//TILE)*TILE).groupby(['site','tile_row','tile_col']):
 row=int(row);col=int(col);p=DGC/f'{s}_{row}_{col}.parquet'
 if p.is_file() and not FORCE_REBUILD:q=pd.read_parquet(p)
 else:q=degradation_tile(s,row,col,g);atomic_parquet(p,q)
 assert q[['site','row','col']].reset_index(drop=True).equals(g[['site','row','col']].reset_index(drop=True))
 parts.append(q);print(s,row,col,'JRC pixels',len(q),flush=True)
DJ=J9.merge(pd.concat(parts,ignore_index=True),on=['site','row','col'],validate='one_to_one')
assert len(DJ)==len(J9)
atomic_parquet(O4/'JRC_native_degradation.parquet',DJ)
def add_blocks(d,tag):
 q=d.copy();q['block500']=''
 for s,g in q.groupby('site'):
  xy=gpd.GeoSeries(gpd.points_from_xy(g.x_native,g.y_native),index=g.index,crs=GRIDS[s,tag][0]).to_crs(CRS)
  bx=np.floor(xy.x.to_numpy()/500).astype(int);by=np.floor(xy.y.to_numpy()/500).astype(int)
  q.loc[g.index,'block500']=[f'{x}_{y}' for x,y in zip(bx,by)]
 return q
HH=add_blocks(H,'Hansen');JJ=add_blocks(DJ,'JRC')
specs=[(HH,'age_since_loss','Hansen detected loss'),(JJ,'age_since_deforestation','JRC deforestation'),(JJ,'age_since_regrowth','JRC regrowth onset'),(JJ,'age_since_degradation','JRC degradation onset')]
tables=[]
for d,col,label in specs:
 q=d[d.history_valid&d[col].notna()].copy()
 assert (q[col]>0).all() and np.allclose(q[col],q[col].astype(int))
 q['age']=q[col].astype(int)
 a=q.groupby(['site','age']).agg(n=('cover_pp','size'),blocks=('block500','nunique'),mean=('cover_pp','mean'),median=('cover_pp','median'),q25=('cover_pp',lambda z:z.quantile(.25)),q75=('cover_pp',lambda z:z.quantile(.75))).reset_index()
 a['variable']=col;a['sufficient_support']=(a.n>=30)&(a.blocks>=5);tables.append(a)
 maxage=max(1,q.age.max()) if len(q) else 1
 fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  g=q[q.site.eq(s)];z=a[a.site.eq(s)]
  ax.scatter(g.age,g.cover_pp,s=3,alpha=.07,color='grey',rasterized=True)
  for robust,zz in z.groupby('sufficient_support'):
   ax.errorbar(zz.age,zz['median'],yerr=[zz['median']-zz.q25,zz.q75-zz['median']],fmt='o' if robust else 'o',mfc='C0' if robust else 'none',mec='C0',ecolor='C0',alpha=1 if robust else .45,label='Median / IQR' if robust else 'Sparse age: median / IQR')
  if len(z):ax.scatter(z.age,z['mean'],marker='x',s=25,color='C1',label='Mean')
  ax.set(title=f'{s} | n={len(g):,}',xlabel='Years since '+label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  if len(z):ax.legend(fontsize=7)
 fig.savefig(O4/f'{col}_annual.png',dpi=200);plt.show();plt.close(fig)
summary=pd.concat(tables,ignore_index=True);summary.to_csv(O4/'annual_age_summary.csv',index=False);display(summary.round(2))
q=DJ.groupby(['site','degradation_status'],dropna=False).cover_pp.agg(n='size',mean='mean',median='median').reset_index()
q.to_csv(O4/'degradation_status_summary.csv',index=False);display(q.round(2))
# Strictly descriptive sensitivity: degradation-only history (no mapped deforested state ever)
# is not inferred from absent dated transition; current analysis reports all dated degradation events.
atomic_json(O4/'completion.json',dict(complete=True,n=len(DJ),source=str(OUT),definition='latest observed 1/4 to 2 transition; annual IQR not CI'))
print('04 COMPLETE',O4)


## 05 — Separate 50 m HLS spectral-level versus dynamics ablation
This is a methodological sensitivity test, not the native-grid ecological chronosequence. Reuses exact matched NB08bis Block 14 50 m reference, same spatial folds, 100 m exclusion, LOFO and equal-forest HGB settings. Test whether six long-term medians of annual spectral observations explain the HLS benefit, versus recent-three-year medians, variability/change descriptors, both, and the full HLS set. Include baseline S2+EMB and baseline+long-term medians / baseline+full HLS. Full available observation windows differ by survey year/site; this is a limitation of the retrospective comparison. No new HLS or UAV extraction; caches kept separate. Paired block bootstrap conditional on fixed predictions and four forests, not regional inference.

In [ ]:
#@title 05 — HLS spectral medians versus temporal dynamics (cached matched 50 m)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn import __version__ as SK_VERSION
from scipy.spatial import cKDTree
from threadpoolctl import threadpool_limits
from sklearn.metrics import roc_auc_score
threadpool_limits(limits=1)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()
def load_hls_cohort():
 ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
 QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
 EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
 Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
 S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
 FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
 ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
 BASE=S2+EMB;BASE_NAME='M3_S2_EMB__HGB__forest_equal'
 KEYS=['site','mosaic','row','col'];MIN_SUPPORT=.90
 SATROOT=ROOT/'Outputs/08_satellite_reference';NB08_OUT=ROOT/'Outputs/08_satellite_science_v5';EXPL=ROOT/'Outputs/08bis_abundance_exploration_v5'
 candidates=[]
 for p in sorted(SATROOT.glob('native_50m_reference_*.parquet')):
  sig=p.stem.removeprefix('native_50m_reference_');cf=NB08_OUT/sig/'config.json'
  if not cf.is_file():continue
  c=json.loads(cf.read_text());tc=SATROOT/'uav_targets'/c['target_id']/'config.json'
  if not tc.is_file() or digest(c)[:20]!=sig or c['features']!=FEATURE_SETS:continue
  t=json.loads(tc.read_text())
  if t['model']!=M['model_sha'] or t['policy']!=M['policy_sha'] or t['support']!=MIN_SUPPORT:continue
  old={a['name']:a for a in t['mosaics']}
  if set(old)!={a['name'] for a in M['mosaics']}:continue
  valid=True
  for a in M['mosaics']:
   f=NB07/'reconstructed_mosaics'/Path(a['output']).name
   if not f.is_file() or old[a['name']]['digest']!=a['digest'] or old[a['name']]['size']!=f.stat().st_size:valid=False;break
   with rasterio.open(f) as ds:
    if ds.tags().get('signature')!=a['digest'] or list(ds.bounds)!=old[a['name']]['bounds']:valid=False;break
  if not valid:continue
  d=pd.read_parquet(p)
  if dataframe_sha(d[KEYS+['x','y','target_fraction']+ALL_FEATURES])==c['source_sha256']:candidates.append((p,c,d))
 assert candidates,'Complete NB08 reference first'
 assert len({c['source_sha256'] for _,c,_ in candidates})==1,'Multiple distinct references; select explicitly'
 REF,CONFIG,D=candidates[0];SOURCE_SHA=CONFIG['source_sha256']
 SAT_ID=CONFIG['satellite_id'];SAT_DIR=SATROOT/'satellite_10m'/SAT_ID
 TARGET_DIR=SATROOT/'uav_targets'/CONFIG['target_id']
 assert len(D)==CONFIG['n'] and set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any()
 assert D.target_fraction.between(0,1).all() and np.isfinite(D[BASE].to_numpy(float)).all()
 found=[]
 for p in sorted((NB08_OUT/'within_landscape_50m').glob('*/heldout_predictions.parquet')):
  c=json.loads((p.parent/'config.json').read_text());z=pd.read_parquet(p)
  if c.get('version')!='within_landscape_v2' or c.get('buffer')!=100 or c.get('block')!=500 or BASE_NAME not in z:continue
  cols=['site','row','col','x','y','target_fraction']
  if not z[cols].equals(D[cols]):continue
  used=sorted(set(sum(c['features'].values(),[])))
  if dataframe_sha(D.assign(fold21=z.fold21.to_numpy())[['site','x','y','target_fraction','fold21']+used])==c['source']:found.append((p,z))
 assert found,'Complete NB08 spatial holdout first'
 BASE_P,OLD=found[0];M21=D.assign(block21=OLD.block21.to_numpy(),fold21=OLD.fold21.to_numpy())
 
 foldsha=dataframe_sha(M21[['site','row','col','fold21']])
 candidates=[]
 for p in EXPL.glob('*/14_long_history/*/hansen_mask_corrected_v2/features_50m.parquet'):
  c=json.loads((p.parent/'config.json').read_text())
  if c['parent']['source']==SOURCE_SHA and c['parent']['fold']==foldsha:candidates.append(p)
 assert len(candidates)==1,candidates
 F=pd.read_parquet(candidates[0])
 assert len(F)==len(M21) and not F.duplicated(KEYS).any()
 d=M21.merge(F,on=KEYS,validate='one_to_one')
 assert len(d)==len(M21)
 return d[d.eligible14].reset_index(drop=True),BASE,str(candidates[0])
 
d5,BASE5,source5=load_hls_cohort()
assert set(d5.site)==set(SITES) and d5.groupby('site').fold21.nunique().eq(5).all()
h=[c for c in d5 if c.startswith('H14_')]
level=[c for c in h if c.endswith('_median')]
recent=[c for c in h if c.endswith('_recent3')]
dyn=[c for c in h if c.endswith(('_sd','_range','_trend','_largest_drop','_drop_age','_decline_fraction','_recovery_slope'))]
ext=[c for c in h if c.endswith(('_min','_max'))]
assert len(level)==len(recent)==6 and set(h)==set(level+recent+dyn+ext),(level,recent,set(h)-set(level+recent+dyn+ext))
# Dynamics include variability summaries; not all features are time-order dependent.
arms={'baseline':BASE5,'longterm_median':level,'recent3_median':recent,'dynamics':dyn,'median_dynamics':level+dyn,'full_HLS':h,'baseline_plus_median':BASE5+level,'baseline_plus_full_HLS':BASE5+h}
model=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
seed=20261123;cols=['site','row','col','x','y','target_fraction','block21','fold21']
cfg=dict(version='HLS_level_dynamics_v1',source=source5,sha=dataframe_sha(d5[cols+list(dict.fromkeys(BASE5+h))]),arms=arms,model=model,seed=seed,sklearn=SK_VERSION,buffer=100)
O5=ROOT/'Outputs/09_ecology_v5'/'05_HLS_ablation'/digest(cfg)[:20];O5.mkdir(parents=True,exist_ok=True);atomic_json(O5/'config.json',cfg)
def splits5():
 xy=d5[['x','y']].to_numpy();site=d5.site.to_numpy();fold=d5.fold21.to_numpy()
 for k in range(5):
  te=np.flatnonzero(fold==k);tr=np.flatnonzero(fold!=k);keep=np.ones(len(tr),bool)
  for s in SITES:
   a=np.flatnonzero(site[tr]==s);b=te[site[te]==s]
   if len(a) and len(b):keep[a]=cKDTree(xy[b]).query(xy[tr[a]])[0]>=100
  assert len(te) and keep.any()
  yield k,tr[keep],te
pred={}
for protocol in ['spatial','LOFO']:
 p=d5[cols].copy()
 sp=list(splits5()) if protocol=='spatial' else [(s,np.flatnonzero(d5.site.ne(s)),np.flatnonzero(d5.site.eq(s))) for s in SITES]
 for k,tr,te in sp:
  g=d5.iloc[tr];sw=1/g.groupby('site').site.transform('size').to_numpy(float);sw/=sw.mean()
  for name,features in arms.items():
   f=O5/f'{protocol}_{k}_{name}.npz'
   if f.is_file() and not FORCE_REBUILD:
    with np.load(f,allow_pickle=False) as z:assert np.array_equal(z['test'],te);v=z['pred']
   else:
    m=HistGradientBoostingRegressor(random_state=seed,**model);m.fit(g[features].to_numpy(np.float32),g.target_fraction.to_numpy(),sample_weight=sw)
    v=np.clip(m.predict(d5.iloc[te][features].to_numpy(np.float32)),0,1)
    tmp=f.with_suffix('.partial')
    with open(tmp,'wb') as z:np.savez_compressed(z,test=te,pred=v)
    os.replace(tmp,f)
   assert len(v)==len(te) and np.isfinite(v).all()
   p.loc[te,name]=v;print(protocol,k,name,flush=True)
 assert p[list(arms)].notna().all().all()
 atomic_parquet(O5/f'{protocol}_predictions.parquet',p);pred[protocol]=p
def metrics5(y,z,w):
 y=np.asarray(y);z=np.asarray(z);w=np.asarray(w,float);w/=w.sum()
 e=z-y;var=np.sum(w*(y-np.sum(w*y))**2)
 h=y>=.4
 return dict(R2=1-np.sum(w*e**2)/var if var>0 else np.nan,MAE_pp=100*np.sum(w*abs(e)),RMSE_pp=100*np.sqrt(np.sum(w*e**2)),bias_pp=100*np.sum(w*e),AUC40=roc_auc_score(h,z,sample_weight=w) if h.any() and (~h).any() else np.nan)
summary=[];per=[];boot=[]
pairs=[('longterm_median','recent3_median'),('dynamics','longterm_median'),('median_dynamics','longterm_median'),('full_HLS','longterm_median'),('full_HLS','median_dynamics'),('baseline_plus_median','baseline'),('baseline_plus_full_HLS','baseline_plus_median')]
for protocol,p in pred.items():
 w=1/p.groupby('site').site.transform('size').to_numpy(float);w/=w.sum()
 den=np.mean([g.target_fraction.var(ddof=0) for _,g in p.groupby('site')])
 for name in arms:
  m=metrics5(p.target_fraction,p[name],w.copy())
  auc=[]
  for s,g in p.groupby('site'):
   b=metrics5(g.target_fraction,g[name],np.ones(len(g)));auc.append(b['AUC40']);per.append(dict(protocol=protocol,model=name,site=s,n=len(g),**b))
  summary.append(dict(protocol=protocol,model=name,n=len(p),**m,within_forest_R2=1-(m['RMSE_pp']/100)**2/den,mean_site_AUC40=np.nanmean(auc)))
 for name,ref in pairs:
  delta=100*(abs(p[name]-p.target_fraction)-abs(p[ref]-p.target_fraction))
  rng=np.random.default_rng(seed);dist=[]
  for s,g in p.assign(delta=delta).groupby('site'):
   q=g.groupby('block21').delta.agg(['sum','count']).to_numpy()
   v=q[rng.integers(0,len(q),(1000,len(q)))].sum(1);dist.append(v[:,0]/v[:,1])
  v=np.mean(dist,axis=0)
  boot.append(dict(protocol=protocol,model=name,reference=ref,delta_MAE_pp=np.sum(w*delta),ci_low=np.quantile(v,.025),ci_high=np.quantile(v,.975)))
S=pd.DataFrame(summary);P=pd.DataFrame(per);B=pd.DataFrame(boot)
S.to_csv(O5/'summary.csv',index=False);P.to_csv(O5/'by_forest.csv',index=False);B.to_csv(O5/'paired_bootstrap.csv',index=False)
display(S.round(3));display(B.round(3))
atomic_json(O5/'completion.json',dict(complete=True,n=len(d5),config=cfg))
print('05 COMPLETE',O5)


## 06 — Final HLS ecological analysis on existing 30 m export pixels (offline)
The earlier scene-native experiment revealed that some HLS scenes within the same MGRS tile have incompatible projection metadata or grids. Do not assume they are identical. Reuse the successfully extracted NB08bis Block 14 annual HLS GeoTIFFs on their verified common EPSG:32635 30 m grid. These were already spatially harmonized by Earth Engine; no extra spectral resampling, new downloads, scene inventory or API calls. Aggregate the original NB07 UAV binary maps directly to each existing raster pixel, using 30×30 subpixel samples and ≥90% valid support inside the inward-buffered ROI, best-supported mosaic, deterministic tie-break. Restrict analysis to cached HLS tile footprints; report coverage and missing years. Distinct from the exact native Hansen/JRC grids. Full available history and a common six-year pre-survey window, annual median spectra and indices, ≥3 clear observations per year and ≥5 valid years per summary. Interpret spectral decline/recovery descriptively, not as verified disturbance.

In [ ]:
#@title 06A — Read cached annual HLS tiles and aggregate UAV directly to their 30 m pixels
HSTART=2013;HMINOBS=3;HMINYEARS=5;HCOMMON=6;HSUB=30
HBANDS=['NDVI','NDMI','NBR','NIR','SWIR1','SWIR2']
EXPL=ROOT/'Outputs/08bis_abundance_exploration_v5'
# Resolve the completed corrected Block 14 run associated with the latest verified native ecology source.
candidates=[]
for p in EXPL.glob('*/14_long_history/*/hansen_mask_corrected_v2/features_50m.parquet'):
 cf=p.parent/'config.json'
 if not cf.is_file():continue
 c=json.loads(cf.read_text())['parent']
 if c.get('years')!=YEARS or c.get('grid')!=30 or c.get('crs')!=CRS or c.get('tile_m')!=1500:continue
 # Match the reference config's source hash and UAV model/policy/mosaic identities.
 sat=ROOT/'Outputs/08_satellite_reference'
 refs=[]
 for ref in sat.glob('native_50m_reference_*.parquet'):
  rc=ROOT/'Outputs/08_satellite_science_v5'/ref.stem.removeprefix('native_50m_reference_')/'config.json'
  if not rc.is_file():continue
  d=json.loads(rc.read_text())
  if d.get('source_sha256')!=c.get('source'):continue
  tc=sat/'uav_targets'/d['target_id']/'config.json'
  if not tc.is_file():continue
  t=json.loads(tc.read_text())
  if t.get('model')!=M['model_sha'] or t.get('policy')!=M['policy_sha']:continue
  old={a['name']:a for a in t['mosaics']}
  if set(old)!={a['name'] for a in meta}:continue
  if not all(old[a['name']]['digest']==a['signature'] and old[a['name']]['size']==a['size'] and old[a['name']]['bounds']==a['bounds'] for a in meta):continue
  refs.append(ref)
 if refs and (p.parent.parent/'tiles').is_dir() and (p.parent.parent/'annual_HLS_50m.parquet').is_file():candidates.append((p,c))
assert len(candidates)==1,('Expected one matching completed HLS cache',[(str(p),c.get('source')) for p,c in candidates])
fp,recipe=candidates[0];T14=fp.parent.parent/'tiles'
import re
files={}
for f in sorted(T14.glob('*_HLS_*.tif')):
 m=re.fullmatch(r'(.+)_(-?\d+)_(-?\d+)_HLS_(\d{4})\.tif',f.name)
 if not m:continue
 s,tx,ty,year=m.groups();tx=int(tx);ty=int(ty);year=int(year)
 if s in SITES and HSTART<=year<YEARS[s]:
  files[s,tx,ty,year]=f
tiles=sorted({k[:3] for k in files})
assert set(k[0] for k in tiles)==set(SITES)
for s,tx,ty in tiles:
 assert all((s,tx,ty,y) in files for y in range(HSTART,YEARS[s])),('Incomplete annual raster series',s,tx,ty)
HCFG=dict(version='HLS_cached_export30_v1',source=str(fp),recipe=recipe,mosaics=meta,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),support=SUPPORT,sub=HSUB,common_years=HCOMMON,min_years=HMINYEARS,min_obs=HMINOBS)
HOUT=ROOT/'Outputs/09_ecology_v5'/'06_cached_HLS_30m'/digest(HCFG)[:20];HOUT.mkdir(parents=True,exist_ok=True)
HCACHE=HOUT/'tiles';HCACHE.mkdir(exist_ok=True);atomic_json(HOUT/'config.json',HCFG)
def tile30(s,tx,ty,stack):
 tr=Affine(30,0,tx*1500,0,-30,(ty+1)*1500);size=50*HSUB;hi=tr*Affine.scale(1/HSUB,1/HSUB)
 inside=rasterize([(GEOM[s],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(50,HSUB,50,HSUB).sum((1,3))
 best=np.zeros((50,50),np.int32);pos=best.copy();src=np.full(best.shape,-1,np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  with WarpedVRT(stack[name],crs=CRS,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1));better=nv>best
  best[better]=nv[better];pos[better]=npix[better];src[better]=i
 keep=best>=math.ceil(SUPPORT*HSUB*HSUB)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 u=pd.DataFrame(dict(site=s,row=rr-50*(ty+1),col=cc+50*tx,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(HSUB*HSUB),cover_pp=100*pos[keep]/best[keep],mosaic=[MOS[s][i][0] for i in src[keep]]))
 annual=[]
 for y in range(HSTART,YEARS[s]):
  f=files[s,tx,ty,y];side=json.loads(f.with_suffix('.json').read_text())
  with rasterio.open(f) as ds:
   assert ds.shape==(50,50) and ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.transform.almost_equals(tr)
   assert list(ds.descriptions)==HBANDS+['N_CLEAR'] and ds.tags().get('recipe')==side['recipe']
   a=ds.read()[:,rr,cc].T
  count=a[:,-1];z=a[:,:6];valid=(count>=HMINOBS)&np.isfinite(z).all(1)&(z!=-9999).all(1)
  d=u[['site','row','col']].copy();d['year']=y;d['n_clear']=count;d['valid_year']=valid
  for k,b in enumerate(HBANDS):d[b]=np.where(valid,z[:,k],np.nan)
  annual.append(d)
 return u,pd.concat(annual,ignore_index=True)
uref=[];annual=[]
for s in SITES:
 st=[k for k in tiles if k[0]==s]
 with ExitStack() as es:
  stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
  for i,(_,tx,ty) in enumerate(st):
   up=HCACHE/f'{s}_{tx}_{ty}_UAV.parquet';ap=HCACHE/f'{s}_{tx}_{ty}_annual.parquet'
   if up.is_file() and ap.is_file() and not FORCE_REBUILD:u=pd.read_parquet(up);a=pd.read_parquet(ap)
   else:u,a=tile30(s,tx,ty,stack);atomic_parquet(up,u);atomic_parquet(ap,a)
   assert len(a)==len(u)*(YEARS[s]-HSTART)
   if len(u):uref.append(u);annual.append(a)
   print(s,i+1,'/',len(st),'pixels',len(u),flush=True)
U6=pd.concat(uref,ignore_index=True);A6=pd.concat(annual,ignore_index=True);K6=['site','row','col']
assert set(U6.site)==set(SITES) and not U6.duplicated(K6).any() and not A6.duplicated(K6+['year']).any()
assert U6.support.ge(SUPPORT).all() and U6.cover_pp.between(0,100).all()
atomic_parquet(HOUT/'HLS_native_UAV_reference.parquet',U6);atomic_parquet(HOUT/'HLS_native_annual.parquet',A6)
q=A6.groupby(K6).valid_year.sum().rename('valid_years').reset_index().merge(U6,on=K6,validate='one_to_one')
qa=q.groupby('site').agg(n_pixels=('cover_pp','size'),mean_UAV_support=('support','mean'),mean_valid_years=('valid_years','mean'),n_ge5_years=('valid_years',lambda z:(z>=HMINYEARS).sum())).reset_index()
qa.to_csv(HOUT/'coverage_QA.csv',index=False);display(qa)
atomic_json(HOUT/'extraction_complete.json',dict(complete=True,n_pixels=len(U6),n_pixel_years=len(A6)))
print('06A COMPLETE — cached 30m grids, no EE calls',HOUT)


In [ ]:
#@title 06B — Simple 30 m export-pixel scatterplots and fixed-window sensitivity
import warnings
from scipy.stats import spearmanr
assert json.loads((HOUT/'extraction_complete.json').read_text())['complete']
U6=pd.read_parquet(HOUT/'HLS_native_UAV_reference.parquet');A6=pd.read_parquet(HOUT/'HLS_native_annual.parquet')
K6=['site','row','col'];F6=[]
def features6(g,s,window):
 years=np.arange(HSTART if window=='full' else YEARS[s]-HCOMMON,YEARS[s])
 u=U6[U6.site.eq(s)].reset_index(drop=True);idx=pd.MultiIndex.from_frame(u[K6])
 a=g[g.year.isin(years)]
 out=u.copy();out['window']=window;out['n_window_years']=len(years)
 for band in HBANDS:
  z=a.pivot(index=K6,columns='year',values=band).reindex(index=idx,columns=years).to_numpy(float)
  ok=np.isfinite(z);n=ok.sum(1);eligible=n>=HMINYEARS
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',category=RuntimeWarning)
   med=np.nanmedian(z,axis=1);avg=np.nanmean(z,axis=1);sd=np.nanstd(z,axis=1)
  dt=years[None,:]-np.divide((ok*years[None,:]).sum(1),n,out=np.zeros(len(n),float),where=n>0)[:,None]
  den=(ok*dt**2).sum(1);trend=np.divide((dt*np.where(ok,z,0)).sum(1),den,out=np.full(len(z),np.nan),where=den>0)
  # Adjacent valid annual observations only; no interpolation through gaps.
  delta=np.diff(z,axis=1);adj=np.isfinite(delta)
  k=np.argmin(np.where(adj,delta,np.inf),axis=1)
  drop=delta[np.arange(len(z)),k];drop=np.where(adj.any(1),np.maximum(-drop,0),np.nan)
  rec=np.full(len(z),np.nan)
  if band in ['NDVI','NDMI','NBR']:
   # A decline of >=0.05 index units is a descriptive spectral event, not validated disturbance.
   for i in np.flatnonzero(eligible&(drop>=.05)):
    trough=k[i]+1;ix=np.flatnonzero(ok[i]&(np.arange(len(years))>trough))
    if len(ix):rec[i]=(z[i,ix[-1]]-z[i,trough])/(years[ix[-1]]-years[trough])
  out[band+'_valid_years']=n
  for name,v in [('median',med),('mean',avg),('sd',sd),('trend',trend),('largest_decline',drop),('recovery_slope',rec)]:
   if name=='recovery_slope' and band not in ['NDVI','NDMI','NBR']:continue
   out[band+'_'+name]=np.where(eligible,v,np.nan)
 return out
for s in SITES:
 g=A6[A6.site.eq(s)]
 for window in ['full','common6']:F6.append(features6(g,s,window))
F6=pd.concat(F6,ignore_index=True);atomic_parquet(HOUT/'HLS_native_features.parquet',F6)
rows=[]
for window in ['full','common6']:
 for band in HBANDS:
  fields=['median','mean','sd','trend','largest_decline']+(['recovery_slope'] if band in ['NDVI','NDMI','NBR'] else [])
  fig,axs=plt.subplots(len(SITES),len(fields),figsize=(3.2*len(fields),3*len(SITES)),sharey=True,squeeze=False,layout='constrained')
  for i,s in enumerate(SITES):
   g=F6[F6.site.eq(s)&F6.window.eq(window)]
   for j,field in enumerate(fields):
    col=band+'_'+field;ax=axs[i,j];q=g[g[col].notna()]
    x=q[col];y=q.cover_pp
    rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
    ax.scatter(x,y,s=3,alpha=.10,rasterized=True)
    ax.set(title=f'{s} | n={len(q):,} | rho={rho:.2f}',xlabel=band+' '+field.replace('_',' '),ylim=(0,100))
    if j==0:ax.set_ylabel('UAV Musanga cover (%)')
    rows.append(dict(window=window,site=s,band=band,statistic=field,n=len(q),n_total=len(g),rho=rho))
  fig.suptitle(f'{band} — {window} historical window')
  fig.savefig(HOUT/f'{band}_{window}_scatter.png',dpi=180);plt.show();plt.close(fig)
stats=pd.DataFrame(rows);stats.to_csv(HOUT/'raw_scatter_summary.csv',index=False)
display(stats.pivot_table(index=['site','band','statistic'],columns='window',values='rho').round(3))
# Compare spectral levels with the same fixed six-year relative window for every forest.
# Means/medians include all valid annual observations; no imputed years, >=5/6 years required.
qa=F6.groupby(['window','site']).agg(n=('cover_pp','size'),eligible=('NBR_median','count'),mean_valid_years=('NBR_valid_years','mean')).reset_index()
qa.to_csv(HOUT/'window_support.csv',index=False);display(qa)
atomic_json(HOUT/'completion.json',dict(complete=True,n_export_grid_pixels=len(U6),n_pixel_years=len(A6),config=HCFG,limitations=['observational spatial chronosequence','unequal full observation windows','spectral event not validated forest disturbance','existing spatially harmonized 30m UTM HLS exports, not original scene grids','UAV-derived estimated cover']))
print('06 CACHED HLS COMPLETE',HOUT)


## 07 — Concise Word summary
Three core figures: disturbance condition boxplots, disturbance-age boxplots, HLS correlation heatmap. Latest compatible completed results only. Fixed output path, overwritten on rerun. No separate figure exports or archive. Existing computational caches and legacy reports are not deleted.


In [ ]:
#@title 07 — Final Word summary (latest only)
%pip -q install python-docx
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from io import BytesIO
E=ROOT/'Outputs/09_ecology_v5'
def done(p):
 f=p/'completion.json'
 return f.is_file() and json.loads(f.read_text()).get('complete',False)
def latest(paths):
 paths=[p for p in paths if done(p)]
 assert paths,'No compatible completed source found'
 return max(paths,key=lambda p:(p/'completion.json').stat().st_mtime_ns)
def compatible(p):
 c=json.loads((p/'completion.json').read_text()).get('config',{})
 return c.get('roi_sha')==hashlib.sha256(roi.wkb).hexdigest() and c.get('support')==SUPPORT and {x['name']:(x['signature'],x['size']) for x in c.get('mosaics',[])}=={x['name']:(x['signature'],x['size']) for x in meta}
N=latest([p for p in E.iterdir() if p.is_dir() and done(p) and (p/'Hansen_native_pixels.parquet').is_file() and (p/'JRC_native_pixels.parquet').is_file() and compatible(p)])
L=latest([p for p in (E/'06_cached_HLS_30m').glob('*') if done(p) and compatible(p)])
H=pd.read_parquet(N/'Hansen_native_pixels.parquet');J=pd.read_parquet(N/'JRC_native_pixels.parquet')
F=pd.read_parquet(L/'HLS_native_features.parquet');C=pd.read_csv(L/'raw_scatter_summary.csv')
H=H[H.history_valid];J=J[J.history_valid]
assert set(H.site)==set(J.site)==set(F.site)==set(SITES)
D4=N/'04_annual_ages_and_degradation_v1';has4=done(D4)
source5=json.loads((L/'config.json').read_text())['source']
matches=[p for p in (E/'05_HLS_ablation').glob('*') if done(p) and json.loads((p/'config.json').read_text()).get('source')==source5]
D5=latest(matches) if matches else None
R7=E/'07_scientific_summary';R7.mkdir(exist_ok=True)
doc=Document();sec=doc.sections[0]
sec.page_width=Inches(8.27);sec.page_height=Inches(11.69)
sec.left_margin=sec.right_margin=Inches(.65);sec.top_margin=sec.bottom_margin=Inches(.65)
normal=doc.styles['Normal'];normal.font.name='Calibri';normal.font.size=Pt(10.5);normal.paragraph_format.space_after=Pt(5)
for name in ['Title','Heading 1','Heading 2']:doc.styles[name].font.color.rgb=RGBColor.from_string('17334D')
def table(head,rows):
 t=doc.add_table(rows=1,cols=len(head));t.style='Light Shading Accent 1'
 for c,v in zip(t.rows[0].cells,head):c.text=str(v)
 for row in rows:
  for c,v in zip(t.add_row().cells,row):c.text=str(v)
def boxes(ax,d,group,order,labels):
 vals=[];pos=[];labs=[]
 for i,(k,l) in enumerate(zip(order,labels),1):
  z=d.loc[d[group].eq(k),'cover_pp'].dropna().to_numpy()
  labs.append(f'{l}\n{len(z):,}'+('*' if 0<len(z)<30 else ''))
  if len(z):vals.append(z);pos.append(i)
 if vals:
  b=ax.boxplot(vals,positions=pos,widths=.6,patch_artist=True,showfliers=False,showmeans=True,meanprops=dict(marker='D',markerfacecolor='#cf7b35',markeredgecolor='white',markersize=4),medianprops=dict(color='#17334d'))
  for p,z in zip(b['boxes'],vals):p.set_facecolor('#a7c4d6' if len(z)>=30 else '#ddd')
 ax.set_xticks(range(1,len(order)+1),labs,fontsize=7);ax.set_ylim(0,100);ax.grid(axis='y',alpha=.2)
def embed(fig,caption):
 buf=BytesIO();fig.savefig(buf,format='png',dpi=220,bbox_inches='tight',facecolor='white');buf.seek(0)
 doc.add_picture(buf,width=Inches(6.9));doc.paragraphs[-1].alignment=WD_ALIGN_PARAGRAPH.CENTER
 doc.add_paragraph(caption,style='Caption');plt.close(fig)
doc.add_heading('Musanga: disturbance and spectral ecology',0)
doc.add_paragraph('NB09 | Scientific synthesis | Baego, MOS, Yangambi and Yoko')
doc.add_heading('1. Disturbance occurrence and forest condition',1)
doc.add_paragraph('Musanga cover is higher in pixels with detected historical tree-cover loss and in mapped degraded forest. The contrast is descriptive; Yangambi has very few dated events.')
rows=[]
for s in SITES:
 q=H[H.site.eq(s)];a=q[q.loss_status.eq('no detected loss')];b=q[q.loss_status.eq('prior loss')]
 rows.append([s,f'{a.cover_pp.mean():.1f}',f'{b.cover_pp.mean():.1f}',f'{len(b):,}'])
table(['Forest','No loss (%)','Prior loss (%)','Prior-loss n'],rows)
fig,axs=plt.subplots(2,4,figsize=(12,5.2),sharey=True,layout='constrained')
for i,s in enumerate(SITES):
 boxes(axs[0,i],H[H.site.eq(s)],'loss_status',['no detected loss','prior loss'],['No loss','Prior loss']);axs[0,i].set_title(s)
 boxes(axs[1,i],J[J.site.eq(s)],'jrc_state',[1,2,3,4],['Undist.','Degrad.','Defor.','Regrowth'])
axs[0,0].set_ylabel('Hansen: cover (%)');axs[1,0].set_ylabel('JRC: cover (%)')
embed(fig,'Figure 1. UAV cover by detected loss and pre-survey JRC condition. Boxes = median/IQR; whiskers = 1.5 IQR; diamond = mean; labels give n; * n<30. Outliers hidden visually only. Survey-year/later loss and non-target JRC states omitted.')
doc.add_page_break()
doc.add_heading('2. Successional age',1)
doc.add_paragraph('Weak or inconsistent monotonic relationships do not support a universal quantitative age indicator. Binned distributions help assess possible nonlinear responses.')
ages=[(H,'age_since_loss','Hansen loss'),(J,'age_since_deforestation','JRC deforestation'),(J,'age_since_regrowth','JRC regrowth')]
if has4:
 g=pd.read_parquet(D4/'JRC_native_degradation.parquet');ages.append((g[g.history_valid],'age_since_degradation','JRC degradation'))
fig,axs=plt.subplots(len(ages),4,figsize=(12,2.0*len(ages)),sharey=True,squeeze=False,layout='constrained')
bins=['1–3','4–7','8–12','13–20','>20'];rows=[]
for i,(d,col,label) in enumerate(ages):
 q=d[d[col].notna()].copy();q['age_bin']=pd.cut(q[col],[0,3,7,12,20,np.inf],labels=bins)
 row=[label]
 for j,s in enumerate(SITES):
  g=q[q.site.eq(s)];ax=axs[i,j];boxes(ax,g,'age_bin',bins,bins)
  if i==0:ax.set_title(s)
  if j==0:ax.set_ylabel(label+' (%)',fontsize=8)
  rho=spearmanr(g[col],g.cover_pp).statistic if len(g)>2 and g[col].nunique()>1 and g.cover_pp.nunique()>1 else np.nan
  row.append(f'{rho:.2f} (n={len(g):,})' if np.isfinite(rho) else f'NA (n={len(g):,})')
 rows.append(row)
embed(fig,'Figure 2. Musanga cover by dated event age (years). No detected event is not age zero. Common age bins; sparse bins are descriptive.')
table(['Event',*SITES],rows)
doc.add_paragraph('Table 2. Spearman correlations using unbinned ages. '+('Degradation-age results included.' if has4 else 'Degradation-age analysis pending.'))
doc.add_page_break()
doc.add_heading('3. Historical spectra and variability',1)
fig,axs=plt.subplots(1,2,figsize=(10,3.8),layout='constrained')
for ax,stat in zip(axs,['median','sd']):
 a=C[(C.window=='common6')&(C.statistic==stat)].pivot(index='band',columns='site',values='rho').reindex(index=['NIR','SWIR1','SWIR2','NDVI','NDMI','NBR'],columns=SITES)
 im=ax.imshow(a,vmin=-1,vmax=1,cmap='RdBu_r',aspect='auto')
 ax.set_xticks(range(4),SITES,rotation=30,ha='right');ax.set_yticks(range(6),a.index);ax.set_title('Historical median' if stat=='median' else 'Interannual SD')
 for i in range(6):
  for j in range(4):
   v=a.iloc[i,j];ax.text(j,i,f'{v:.2f}' if np.isfinite(v) else 'NA',ha='center',va='center',fontsize=9,color='white' if abs(v)>.65 else 'black')
fig.colorbar(im,ax=axs,label='Spearman rho',shrink=.8)
embed(fig,'Figure 3. Within-forest correlations with UAV cover, common six-year window.')
doc.add_paragraph('NIR shows a positive association in all four forests. SWIR reverses sign at Yangambi. Variability is positively associated with cover especially at MOS and Yoko; spectral trends and recovery are less consistent. These are associations, not demonstrated causal mechanisms.')
doc.add_heading('Predictive implications',2)
if D5:
 m=pd.read_csv(D5/'summary.csv');m=m[m.protocol.eq('LOFO')]
 table(['Model','LOFO R²','MAE (pp)'],[[r.model,f'{r.R2:.3f}',f'{r.MAE_pp:.2f}'] for r in m.itertuples()])
else:doc.add_paragraph('Block 05 pending: whether dynamics add predictive value beyond historical medians remains unresolved.')
doc.add_heading('Methods and limitations',2)
doc.add_paragraph(f'{len(H):,} Hansen and {len(J):,} JRC valid-history pixels on native grids; {len(F[F.window.eq("common6")]):,} HLS pixels on the existing harmonized 30 m export grid. UAV-derived binary canopy maps aggregated with ≥90% support. Histories precede the survey year. Spatial autocorrelation, unequal sampling, retrospective mapping and uncertainty in disturbance detection and UAV classification limit inference. Spatial chronosequences are not longitudinal observations.')
out=R7/'NB09_scientific_summary.docx';tmp=R7/'NB09_scientific_summary.partial.docx'
doc.save(tmp);os.replace(tmp,out)
print('07 COMPLETE — latest Word report:',out)
print('Source runs:',N.name,L.name,'04:',has4,'05:',D5 is not None)
